# Deep Agents 第 1 阶段：harness 基础

对应 `codes/deepagents/index.md` 第 1 阶段，讲义见同目录 `README.md`。

先运行下一格加载 `DEEPSEEK_API_KEY`（可放 `codes/langgraph/local.env`）。下面 `# 作答` 已补上参考实现；自己再跑一遍核对即可。


In [ ]:
import os
from pathlib import Path

for p in [Path.cwd(), *Path.cwd().parents]:
    env = p / "codes/langgraph/local.env"
    if env.is_file():
        for line in env.read_text().splitlines():
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

from langchain.chat_models import init_chat_model
model = init_chat_model(f"deepseek:{os.environ.get('DEEPSEEK_MODEL', 'deepseek-v4-flash')}")

## 1. 最小 deep agent 与它的工具集

用 `create_deep_agent(model=model)` 建一个代理，让它「把三条 Python 并发要点写进 /notes.md」。

1. 打印 `out.keys()` 和 `out["files"]`。
2. 文件在磁盘上吗？为什么？
3. 工具列表里有没有 `write_todos`？（提示：看 0.7 发布说明）

In [ ]:
from deepagents import create_deep_agent

# 作答
agent = create_deep_agent(model=model)
out = agent.invoke({
    "messages": [{"role": "user", "content": "把三条 Python 并发要点写进 /notes.md"}]
})

print("keys:", sorted(out.keys()))
print("files:", out.get("files"))

# 1) files 在 state 里（默认 StateBackend），不在磁盘
# 2) 0.7 起默认工具集没有 write_todos（opt-in）
tool_names = set()
for m in out["messages"]:
    for tc in getattr(m, "tool_calls", None) or []:
        tool_names.add(tc["name"])
# 看绑定过的工具：从图节点 / 中间件也行；这里用一次空 invoke 前的 bind 不够直观，
# 直接再看默认 agent 的工具名（create 时已装好）：
bound = None
for _, node in agent.get_graph().nodes.items():
    pass
# 更稳：再造一个假模型探测，见 check.py；或读 messages 里出现过的工具名
print("tool_calls seen in run:", sorted(tool_names))
print("write_todos in this run?", "write_todos" in tool_names)
print("on disk?", Path("/notes.md").exists())  # 期望 False：虚拟 FS


## 2. 打开计划工具

给第 1 题的代理加上 `TodoListMiddleware`，换一个多步任务（例如「对比 asyncio / threading / multiprocessing，分三节写进 /report.md」）。

1. 用 `agent.stream(..., stream_mode="updates")` 观察 `todos` 的状态变化。
2. 和不加时比，模型调用次数和最终文件有什么差别？

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

# 作答
agent = create_deep_agent(model=model, middleware=[TodoListMiddleware()])

print("--- stream updates (todos) ---")
for chunk in agent.stream(
    {"messages": [{"role": "user", "content":
        "对比 asyncio / threading / multiprocessing，分三节写进 /report.md。"
        "先用 write_todos 列出计划，再逐节完成。"}]},
    stream_mode="updates",
):
    # updates: {node_name: state_delta}
    for node, delta in chunk.items():
        if isinstance(delta, dict) and "todos" in delta:
            print(node, "todos ->", delta["todos"])

out = agent.invoke({
    "messages": [{"role": "user", "content":
        "对比 asyncio / threading / multiprocessing，分三节写进 /report.md。"
        "先用 write_todos 列出计划，再逐节完成。"}]
})
print("final todos:", out.get("todos"))
print("files keys:", list((out.get("files") or {}).keys()))
# 对比第 1 题：多了 todos 通道；模型通常先 write_todos 再写文件，调用次数更多。


## 3. 子代理与上下文隔离

定义一个 `critic` 子代理（`name` / `description` / `system_prompt`），主代理先写草稿，再派 `critic` 审阅。

1. 在 `out["messages"]` 里找到 `task` 对应的 `ToolMessage`，它包含子代理的哪部分输出？
2. 子代理的 `system_prompt` 出现在父代理消息里了吗？
3. 把 `description` 改成一个含糊的词，再跑一次，主代理还会派活吗？

In [ ]:
# 作答
critic = {
    "name": "critic",
    "description": "审阅技术草稿：挑漏洞、缺出处、结构问题，只回传审阅结论",
    "system_prompt": "你是苛刻的技术审稿人。只输出：问题列表 + 是否通过。不要复述全文。",
}

agent = create_deep_agent(model=model, subagents=[critic])
out = agent.invoke({
    "messages": [{"role": "user", "content":
        "先把『Python GIL 对线程的影响』写成短草稿到 /draft.md，"
        "再派 critic 子代理审阅，根据审阅意见改一版。"}]
})

from langchain_core.messages import ToolMessage
for m in out["messages"]:
    if isinstance(m, ToolMessage) or getattr(m, "type", None) == "tool":
        name = getattr(m, "name", None) or ""
        if name == "task" or "task" in str(getattr(m, "content", ""))[:20]:
            print("--- task ToolMessage (prefix) ---")
            print(str(m.content)[:500])
            break
else:
    # fallback: 任何 tool 消息里找子代理结论
    tool_msgs = [m for m in out["messages"] if getattr(m, "type", None) == "tool"]
    print("tool messages:", len(tool_msgs))
    if tool_msgs:
        print(str(tool_msgs[0].content)[:500])

parent_text = "\n".join(str(getattr(m, "content", "")) for m in out["messages"])
print("parent sees critic system_prompt?", "苛刻的技术审稿人" in parent_text)

# description 含糊时：主模型常不派活
vague = {"name": "critic", "description": "助手", "system_prompt": "你是审稿人。"}
agent2 = create_deep_agent(model=model, subagents=[vague])
out2 = agent2.invoke({
    "messages": [{"role": "user", "content": "写短草稿到 /draft.md，必要时可派子代理审阅。"}]
})
task_calls = []
for m in out2["messages"]:
    for tc in getattr(m, "tool_calls", None) or []:
        if tc.get("name") == "task":
            task_calls.append(tc)
print("vague description task calls:", len(task_calls))


## 4. 换 backend：长期记忆

用 `CompositeBackend` 把 `/memories/` 路由到 `StoreBackend`（`InMemoryStore`），其余仍走 `StateBackend`。

1. 第一个 `thread_id` 里让代理把「用户偏好：回答简洁」写进 `/memories/pref.md`。
2. 换一个新 `thread_id`，让代理先读这个文件再回答。哪些文件跨线程还在？

注意：用 `checkpointer=MemorySaver()` 才有 thread 概念。

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import MemorySaver

# 作答
store = InMemoryStore()
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda rt: ("user-demo",), store=store)},
)
checkpointer = MemorySaver()
agent = create_deep_agent(
    model=model,
    backend=backend,
    store=store,
    checkpointer=checkpointer,
)

cfg1 = {"configurable": {"thread_id": "t1"}}
out1 = agent.invoke(
    {"messages": [{"role": "user", "content":
        "把『用户偏好：回答简洁』写进 /memories/pref.md，再随手写个 /scratch.md 记临时想法。"}]},
    cfg1,
)
print("t1 files:", list((out1.get("files") or {}).keys()))
print("store after t1:", store.search(("user-demo",)))

cfg2 = {"configurable": {"thread_id": "t2"}}
out2 = agent.invoke(
    {"messages": [{"role": "user", "content":
        "先读 /memories/pref.md，按里面的偏好用一句话介绍你自己；"
        "顺便说说你能不能读到 /scratch.md。"}]},
    cfg2,
)
print("t2 files:", list((out2.get("files") or {}).keys()))
print("final:", out2["messages"][-1].content)
# /memories/* 跨 thread 仍在（StoreBackend）；/scratch.md 只在 t1 的 state 里。


## 5. 对比题（答案）

1. **同一种底层，不同一层包装。** `create_deep_agent` 返回的仍是 LangGraph `CompiledStateGraph`，图结构本质还是「模型 ↔ 工具」循环（和 `create_agent` / 第 6 阶段预置 ReAct 一类）。差别在 **中间件 / harness 层**：默认装好了虚拟文件系统、`task` 子代理、摘要、悬空 tool call 修补等，不是另画一张完全不同的图。

2. **固定三步报表 → 手写 LangGraph（或薄 `create_agent`）**；**调研陌生领域 → `create_deep_agent`**。前者步骤已知、要精确控制边与状态；后者步数未知，需要外存文件、可选计划、子代理隔离上下文。

3. **`write_todos` 不执行业务，只把计划写入 state 的 `todos` 通道。** 每轮模型都能看见「还剩什么 / 做到哪」，等于给注意力一个锚点，减少长程目标漂移。0.7 起默认关掉，是因为强模型上评测提分不明显还多花钱；弱模型、超长任务、或要把进度展示给用户时再 `TodoListMiddleware` opt-in。
